INGESTA DE DATOS

In [0]:
%sql
SELECT
    current_catalog() AS catalogo,
    current_schema() AS esquema;

In [0]:
# ============================================
# CONFIGURACIÓN DE INGESTA
# ============================================

ruta_raw = "/Volumes/workspace/default/vol_raw_nyctaxi"

print(f"Fuente de datos: {ruta_raw}")

In [0]:
# ============================================
# DETECTAR CARGAS DISPONIBLES
# ============================================

archivos_raw = dbutils.fs.ls(ruta_raw)

cargas_disponibles = [
    archivo.name.rstrip("/")
    for archivo in archivos_raw
    if archivo.name.startswith("carga_")
]

cargas_disponibles = sorted(cargas_disponibles)

print("Cargas disponibles:")
for carga in cargas_disponibles:
    print(f"- {carga}")

In [0]:
# ============================================
# LECTURA DE LAS CARGAS DISPONIBLES
# ============================================

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv([
        f"{ruta_raw}/{carga}"
        for carga in cargas_disponibles
    ])
)

print(f"Registros totales leídos: {df.count()}")

df.printSchema()

Creacion de la tabla de control

In [0]:
%sql
CREATE TABLE workspace.default.control_cargas_nyctaxi (
    _id_lote STRING,
    _archivo_origen STRING,
    _fecha_inicio TIMESTAMP,
    _fecha_fin TIMESTAMP,
    _fuente STRING,
    _registros_leidos BIGINT,
    _registros_nuevos BIGINT,
    _registros_duplicados BIGINT,
    _registros_rechazados BIGINT,
    _estado STRING
)
USING DELTA;

Consulta de cargas ya procesadas

In [0]:
# ============================================
# IDENTIFICAR CARGAS YA PROCESADAS
# ============================================

from pyspark.sql import functions as F

df_control = spark.table(
    "workspace.default.control_cargas_nyctaxi"
)

filas_control = (
    df_control
    .select("_archivo_origen")
    .where(F.col("_archivo_origen").isNotNull())
    .collect()
)

cargas_procesadas = [
    fila["_archivo_origen"]
    for fila in filas_control
]

cargas_nuevas = [
    carga
    for carga in cargas_disponibles
    if carga not in cargas_procesadas
]

print("Cargas procesadas:")
print(cargas_procesadas)

print("\nCargas nuevas:")
for carga in cargas_nuevas:
    print(f"- {carga}")

Creando tabla bronce

In [0]:
%sql

CREATE TABLE workspace.default.bronze_nyctaxi (
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    pickup_zip INT,
    dropoff_zip INT,
    _id_lote STRING,
    _fecha_ingesta TIMESTAMP,
    _fecha_proceso DATE,
    _fuente STRING,
    _archivo_origen STRING,
    _hash_registro STRING
)
USING DELTA;

In [0]:
%sql

SELECT COUNT(*) AS registros_bronze
FROM workspace.default.bronze_nyctaxi;

Procesando cargas

In [0]:
# ============================================
# PROCESAR CARGAS NUEVAS
# ============================================

from datetime import datetime
import uuid
from pyspark.sql import functions as F

for carga in cargas_nuevas:

    id_lote = str(uuid.uuid4())
    fecha_inicio = datetime.now()

    print(f"Procesando: {carga}")
    print(f"ID lote: {id_lote}")

    # Leer la carga
    df_carga = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{ruta_raw}/{carga}")
    )

    registros_leidos = df_carga.count()

    print(f"Registros leídos: {registros_leidos}")

    # Columnas utilizadas para generar el hash
    columnas_dato = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "pickup_zip",
        "dropoff_zip"
    ]

    # Agregar metadatos de ingesta
    df_bronze = (
        df_carga
        .withColumn("_id_lote", F.lit(id_lote))
        .withColumn("_fecha_ingesta", F.current_timestamp())
        .withColumn("_fecha_proceso", F.current_date())
        .withColumn("_fuente", F.lit("nyctaxi_simulado"))
        .withColumn("_archivo_origen", F.lit(carga))
        .withColumn(
            "_hash_registro",
            F.sha2(
                F.to_json(
                    F.struct(
                        *[F.col(c) for c in columnas_dato]
                    )
                ),
                256
            )
        )
    )

    # Guardar en Bronze
    (
        df_bronze
        .write
        .mode("append")
        .saveAsTable("workspace.default.bronze_nyctaxi")
    )

    print(f"{carga} cargada correctamente en Bronze")
    print("-" * 50)

Verificando Bronce

In [0]:
%sql

SELECT
    _archivo_origen,
    _id_lote,
    COUNT(*) AS registros
FROM workspace.default.bronze_nyctaxi
GROUP BY
    _archivo_origen,
    _id_lote
ORDER BY _archivo_origen;

Registro de tabla de control de cargas (asumiendo duplicados y errores en ceros por ahora)

In [0]:
# ============================================
# REGISTRAR CARGAS EN TABLA DE CONTROL
# ============================================

from datetime import datetime

for carga in cargas_nuevas:

    # Obtener información de la carga en Bronze
    datos_carga = (
        spark.table("workspace.default.bronze_nyctaxi")
        .filter(F.col("_archivo_origen") == carga)
        .select(
            "_id_lote",
            "_fecha_ingesta",
            "_fuente"
        )
        .limit(1)
        .collect()[0]
    )

    id_lote = datos_carga["_id_lote"]
    fecha_inicio = datos_carga["_fecha_ingesta"]
    fuente = datos_carga["_fuente"]

    registros_leidos = (
        spark.table("workspace.default.bronze_nyctaxi")
        .filter(F.col("_archivo_origen") == carga)
        .count()
    )

    fecha_fin = datetime.now()

    registro_control = spark.createDataFrame(
        [(
            id_lote,
            carga,
            fecha_inicio,
            fecha_fin,
            fuente,
            registros_leidos,
            registros_leidos,
            0,
            0,
            "OK"
        )],
        schema=[
            "_id_lote",
            "_archivo_origen",
            "_fecha_inicio",
            "_fecha_fin",
            "_fuente",
            "_registros_leidos",
            "_registros_nuevos",
            "_registros_duplicados",
            "_registros_rechazados",
            "_estado"
        ]
    )

    (
        registro_control
        .write
        .mode("append")
        .saveAsTable(
            "workspace.default.control_cargas_nyctaxi"
        )
    )

    print(f"Control registrado: {carga}")

Revisando tabla de control de cargas

In [0]:
%sql

SELECT
    _archivo_origen,
    _registros_leidos,
    _registros_nuevos,
    _registros_duplicados,
    _registros_rechazados,
    _estado
FROM workspace.default.control_cargas_nyctaxi
ORDER BY _archivo_origen;